This is a notebook to reproduce plots for the analysis of the cytosol dataset in:

S. von Buelow, K. E. Johansson, K. Lindorff-Larsen. AF-CALVADOS: AlphaFold-guided simulations of multi-domain proteins at the proteome level. Protein Science 2026

The code requires the calvados package (https://github.com/KULL-Centre/CALVADOS) and its dependencies, as well as the packages loaded in #Input.

# Input

In [ ]:
import pandas as pd
import numpy as np
import calvados as cal
import os
import json

import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

from tqdm import tqdm
from tqdm import TqdmWarning

import MDAnalysis as mda

import numba as nb
import seaborn as sns
import math
import copy

from scipy.stats import spearmanr, pearsonr, entropy, linregress

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) 
warnings.filterwarnings("ignore", category=TqdmWarning) 

In [ ]:
plt.rcParams['font.size'] = 6
plt.rcParams['ytick.major.size'] = 2
plt.rcParams['ytick.minor.size'] = 1
plt.rcParams['xtick.major.size'] = 2
plt.rcParams['xtick.minor.size'] = 1
plt.rcParams['xtick.labelsize'] = 6
plt.rcParams['ytick.labelsize'] = 6
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['axes.labelpad'] = 2
plt.rcParams['savefig.pad_inches'] = 0.1
plt.rcParams['figure.dpi'] = 300
plt.rcParams['lines.markersize'] = 3
plt.rcParams['lines.markeredgewidth'] = 0.5
plt.rcParams['lines.linewidth'] = 1.
plt.rcParams['font.serif'] = 'Times New Roman'
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.monospace'] = 'Courier New'

In [ ]:
def plotmat(ax,matrix,cmap=plt.cm.Blues,vmin=0,vmax=1,
            pad=0.05,size="5%",unitlabel='nm'):

    _ = ax.imshow(matrix,cmap=cmap,vmin=vmin,vmax=vmax)
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size=size, pad=pad)
    plt.colorbar(_,cax=cax,label=unitlabel)

def calc_idr(scale, min_scale=0.05):
    high_pass = np.where(scale >= min_scale, 1, 0)
    high_pass = np.sum(high_pass,axis=1)
    idr = np.where(high_pass == 0, 1, 0)
    idr_length = np.sum(idr)
    return idr, idr_length

# Class

In [ ]:
class MDP_Cyto:
    def __init__(
        self,
        dataset,
        csv_file=None,
        sim_folder=None,
        pdb_folder=None,
        colabfold=0,
        k_go=20.,
        pae_shift=0.3,
        pae_width=15.,
        bfac_shift=0.8,
        bfac_width=50.,
        cutoff_restraint=0.9,
    ):
        self.dataset = dataset
        self.csv_file = csv_file

        if sim_folder is None:
            self.sim_folder = f'sims_{self.dataset}'
        else:
            self.sim_folder = sim_folder
        if pdb_folder is None:
            self.pdb_folder = f'pdbs_{self.dataset}'
        else:
            self.pdb_folder = pdb_folder

        self.colabfold = colabfold
        self.df = pd.read_csv(csv_file)
        self.k_go=k_go
        self.pae_shift=pae_shift
        self.pae_width=pae_width
        self.bfac_shift=bfac_shift
        self.bfac_width=bfac_width
        self.cutoff_restraint=0.9

    # def get_param_string(self,k_go=0.,pae_shift=0.,pae_width=0.):
    #     param_string = f'k_{k_go:.2f}_cutoff_{cutoff_restraint:.2f}_bshift_{bfac_shift:.2f}_bwidth_{bfac_width:.2f}_paeshift_{pae_shift:.2f}_paewidth_{pae_width:.2f}'
    #     return param_string

    def calc_rgs(self,min_key=None,max_key=None,
               start=10,stop=None,step=1,
               save=True):
        self.rgs = np.zeros((len(self.df[min_key:max_key])))

        for d_idx, (key, val) in enumerate(self.df[min_key:max_key].iterrows()):
            name = val['uniprot']

            # param_string = self.get_param_string(k_go=k_go,pae_shift=pae_shift,pae_width=pae_width)
            path = f'{self.sim_folder}'
            u = mda.Universe(f'{path}/{name}/top.pdb',f'{path}/{name}/{name}.dcd')
            ag = u.atoms
            rg = cal.analysis.calc_rg(u, ag, start=start, stop=stop, step=step)
            self.rgs[d_idx] = np.mean(rg)
        if save:
            np.save(f'rgs/rgs_{self.dataset}.npy', self.rgs)

    def load_rgs(self):
        self.rgs = np.load(f'rgs/rgs_{self.dataset}.npy')

    def calc_dmaps(self,min_key=None,max_key=None,
               start=10,stop=None,step=1,
               save=True,manual=True):
        for d_idx, (key, val) in tqdm(enumerate(self.df[min_key:max_key].iterrows()),total=len(self.df[min_key:max_key])):
            name = val['uniprot']
            if manual:
                dmap_std = self.calc_dmap_std_manual(name,start=start,stop=stop,step=step)
            else:
                dmap_std = self.calc_dmap_std(name,start=start,stop=stop,step=step)

            if save:
                mapfile = f'dmaps_std/dmap_{name}.npy'
                np.save(mapfile, dmap_std)

    def calc_dmap_std(self, name, start=10,stop=None,step=1):
        path = f'{self.sim_folder}'
        u = mda.Universe(f'{path}/{name}/top.pdb',f'{path}/{name}/{name}.dcd')
        ag = u.atoms
        
        dmaps = []

        for t, ts in enumerate(u.trajectory[start:stop:step]):#,total=nframes):
            dmaps.append(cal.analysis.calc_dmap(ag,ag))
        dmaps = np.array(dmaps)
        dmaps_std = np.std(dmaps,axis=0)
        return dmaps_std

    def calc_dmap_std_manual(self, name, start=10,stop=None,step=1):
        path = f'{self.sim_folder}'
        u = mda.Universe(f'{path}/{name}/top.pdb',f'{path}/{name}/{name}.dcd')
        ag = u.atoms

        box = u.dimensions[:3] / 10.
        coordinates = u.trajectory.timeseries(ag) / 10.
        
        nres = len(ag)
        
        dmap_std_manual = self.calc_dmap_std_numba(coordinates,box,nres,start=start,stop=stop,step=step)
        return dmap_std_manual

    @staticmethod
    @nb.jit(nopython=True)
    def calc_dmap_std_numba(coordinates,box,nres,start=10,stop=None,step=1):
        dmaps_std_manual = np.zeros((nres,nres))
        
        for idx in range(nres):
            for jdx in range(idx,nres):
                x0 = coordinates[idx,start:stop:step]
                x1 = coordinates[jdx,start:stop:step]
                dvec = x1-x0
                dvec_pbc = np.where(dvec<=box/2, dvec, box-dvec)
                d = np.zeros((len(x0)))
                for kdx, dv_pbc in enumerate(dvec_pbc):
                    d[kdx] = math.sqrt(dv_pbc[0]**2 + dv_pbc[1]**2 + dv_pbc[2]**2)
                d_std = np.std(d)
        
                dmaps_std_manual[idx,jdx] = d_std
                dmaps_std_manual[jdx,idx] = d_std
        return dmaps_std_manual
    
    # @nb.jit(nopython=True)
    # def calc_length(dvec):
    #     return math.sqrt(dvec[0]**2 + dvec[1]**2 + dvec[2]**2)
        
    def load_dmap(self,name,k_go=0.,pae_shift=0.,pae_width=0.):
        # param_string = self.get_param_string(k_go=k_go,pae_shift=pae_shift,pae_width=pae_width)
        mapfile = f'dmaps_std/dmap_{name}.npy'
        dmaps_std = np.load(mapfile)
        return dmaps_std

    def calc_cmaps(self,min_key=None,max_key=None,start=10,stop=None,step=1,save=True):
        for d_idx, (key, val) in tqdm(enumerate(self.df[min_key:max_key].iterrows()),total=len(self.df[min_key:max_key])):
            name = val['uniprot']
            cmap = self.calc_cmap(name,start=start,stop=stop,step=step)
            if save:
                mapfile = f'cmaps/cmap_{name}.npy'
                np.save(mapfile, cmap)
    
    def calc_cmap(self, name,
                   start=10,stop=None,step=1):

        path = f'{self.sim_folder}'
        u = mda.Universe(f'{path}/{name}/top.pdb',f'{path}/{name}/{name}.dcd')
        ag = u.atoms

        cmap = cal.analysis.cmap_traj(u,ag,ag,cutoff=1.0,start=start,end=stop,step=step)
        return cmap

    def load_cmap(self,name):
        mapfile = f'cmaps/cmap_{name}.npy'
        cmap = np.load(mapfile)
        return cmap
    
    def load_pae(self,name):
        input_pae = f'{self.pdb_folder}/{name}.json'
        pae = cal.build.load_pae(input_pae,symmetrize=True,colabfold=self.colabfold) / 10. # nm
        return pae

    def load_plddt(self,name):
        # pLDDT
        pdb = f'{self.pdb_folder}/{name}.pdb'
        bfac = cal.build.bfac_from_pdb(pdb)
        bfac_matrix = np.minimum.outer(bfac,bfac)
        return bfac_matrix

    def calc_restraints(self, name, min_scale=0.05):
        comp = cal.cfg.Components(
            fresidues = 'residues_CALVADOS3.csv',
            pdb_folder = self.pdb_folder,
            restraint = True,
            restraint_type = 'go',
        
            bfac_shift = self.bfac_shift, # 0.8, # 0.75,
            bfac_width = self.bfac_width, # 100., #30.,
            cutoff_restraint = self.cutoff_restraint,

            k_go = self.k_go,
            pae_shift = self.pae_shift, #0.6, # 0.3,
            pae_width = self.pae_width,# 15. # 15.,
        
            colabfold = self.colabfold,
        )
        
        comp.reset_components()
        comp.add(name=name)
        
        comp_dict = comp.components['system'][name]
        comp_defaults = comp.components['defaults']
        
        prot = cal.components.Protein(name, comp_dict, comp_defaults)
        
        prot.calc_properties()
        # idr, idr_length = self.calc_idr(prot.scale)
        
        prot.scale = np.where(prot.scale > min_scale, prot.scale, 0)
        prot.scale = np.where(prot.dmap < self.cutoff_restraint, prot.scale, 0)
        
        for idx in range(len(prot.scale)):
            prot.scale[idx,idx] = 0.

        return prot.scale

    def calc_confusion_matrix(self, 
                pae, dmaps_std,
                 pae_cutoff=1.0, dmaps_cutoff=0.35):

        dmaps_std_flat = self.flatten_array(dmaps_std)
        pae_flat = self.flatten_array(pae)

        pae_domain = np.where(pae_flat < pae_cutoff, 1, 0)
        dmaps_std_domain = np.where(dmaps_std_flat < dmaps_cutoff, 1, 0)

        domain_sum = pae_domain + dmaps_std_domain
        domain_diff = dmaps_std_domain - pae_domain

        tp = np.sum(np.where(domain_sum==2,1,0))
        tn = np.sum(np.where(domain_sum==0,1,0))
        fp = np.sum(np.where(domain_diff==1,1,0))
        fn = np.sum(np.where(domain_diff==-1,1,0))

        confusion_matrix = np.array([tp, tn, fp, fn])
        return confusion_matrix

    @staticmethod
    @nb.jit(nopython=True)
    def flatten_array(x):
        xflat = []
        for idx in range(len(x)):
            for jdx in range(len(x)):
                if idx != jdx:
                    xflat.append(x[idx,jdx])
        xflat = np.array(xflat)
        return xflat

    def calc_pae_loss(self, min_key=None, max_key=None, save=True, 
                      pae_cutoff=1.0, dmaps_cutoff=0.35):

        confusion_metrics = np.zeros((len(self.df[min_key:max_key]),4))
        for d_idx, (key, val) in enumerate(self.df[min_key:max_key].iterrows()):
            name = val['uniprot']
            dmaps_std = self.load_dmap(name)
            pae = self.load_pae(name)
            
            confusion_matrix = self.calc_confusion_matrix(
                pae, dmaps_std,                            
                pae_cutoff=pae_cutoff, dmaps_cutoff=dmaps_cutoff)

            confusion_metrics[d_idx] = self.calc_confusion_metric(confusion_matrix)

        metrics, errors = self.bootstrap_pae_loss_error(confusion_metrics)
        
        self.sensitivity = metrics[0]
        self.specificity = metrics[1]
        self.precision = metrics[2]
        self.accuracy = metrics[3]

        self.sensitivity_err = errors[0]
        self.specificity_err = errors[1]
        self.precision_err = errors[2]
        self.accuracy_err = errors[3]

    @staticmethod
    def calc_confusion_metric(cmat):
        tps, tns, fps, fns = cmat
        sensitivity = tps / (tps + fns)
        specificity = tns / (tns + fps)
        precision = tps / (tps + fps)
        accuracy = (tps + tns) / (tps + tns + fps + fns)
        return sensitivity, specificity, precision, accuracy
    
    def bootstrap_pae_loss_error(self, metrics, n_bootstraps=1000):
        """ sensitivity, specificity, precision, accuracy """
        # print(metrics.shape) # requires shape=(len(prot), 4)

        metrics_m = np.mean(metrics, axis=0)
        # print(metrics_m.shape)
        errors = np.zeros((n_bootstraps, 4))
        
        for idx in range(n_bootstraps):
            xs = np.arange(len(metrics),dtype=int)
            p = np.random.choice(xs, size=len(xs), replace=True)
            errors[idx] = np.mean(metrics[p], axis=0)

        errors = np.std(errors, axis=0) # along bootstraps

        return metrics_m, errors

# Load data

## Cyto data

In [ ]:
# Requires folders sims_cytosol and pdbs_cytosol in the current directory.
# This is not used for the remainder of the notebook but can be convenient to analyse the full cytosol set data.

mdp_cyto = MDP_Cyto(
        'cytosol', # 'cytosol', # 'tf',
        csv_file='mdp_cyto.csv',
        sim_folder='sims_cytosol',
        pdb_folder='pdbs_cytosol',
        colabfold=0,
        k_go=15.,
        pae_shift=0.3,
        pae_width=15.,
        bfac_shift=0.8,
        bfac_width=50.,
        cutoff_restraint=0.9,
)
mdp_cyto.df.drop('Unnamed: 0', axis=1, inplace=True)

## Load TF data

In [ ]:
# Requires folders sims_cytosol and pdbs_cytosol in the current directory.

mdp_tf = MDP_Cyto(
        'tf_idrs',
        csv_file='mdp_tf.csv',
        sim_folder='sims_cytosol',
        pdb_folder='pdbs_cytosol',
        colabfold=0,
        k_go=15.,
        pae_shift=0.3,
        pae_width=15.,
        bfac_shift=0.8,
        bfac_width=50.,
        cutoff_restraint=0.9,
)
mdp_tf.df.drop('Unnamed: 0', axis=1, inplace=True)

In [ ]:
# mdp_tf.df

In [ ]:
# Calculate sigma(r) of transcription factor set with recalculate = True
# Requires simulation data sims_cytosol from ERDA cytosol.tar in current directory
# This takes roughly one hour! (Instead, the dmaps_std can be loaded from ERDA cytosol.tar)

min_key = 0
max_key = None

start = 10
stop = None
step = 10 # step = 1 in paper

recalculate = True

if recalculate:
    os.makedirs('dmaps_std',exist_ok=True)
    mdp_tf.calc_dmaps(min_key=min_key,max_key=max_key,start=start,step=step,manual=True)

# Compare PAE and sigma(r)

In [ ]:
mdp = copy.deepcopy(mdp_tf)

# Generate random keys
# xs = np.arange(len(mdp.df))
# keys = np.random.choice(xs,30,replace=False)
# print(keys)

# Result from random selection
keys = [ 599,  506, 1198,  786,  493, 1024,   95, 1280,  425, 1401, 1297,    4,  594,  957,
  622,  306,  980, 1114,  593,  575, 1171, 1077, 1235,  705,  724, 1299,  271, 1011,
 1093, 1012]

# nprots = len(df[min_key:max_key])
nprots = len(keys)

print(f'nprots: {nprots}')

prots_per_row = 4

fig, ax = plt.subplots(nprots//prots_per_row+1,prots_per_row*2,figsize=(prots_per_row*3,2.*3/4*max(1,nprots//prots_per_row)))

nplots=(nprots//prots_per_row+1) * prots_per_row*2
print(f'nplots: {nplots}')

empty = nplots - 2*nprots
print(empty)

for d_idx, (key, val) in tqdm(enumerate(mdp.df.iloc[keys].iterrows()),total=nprots):

    name = val['uniprot']
    # input_pae = f'{mdp.pdb_folder}/{name}.json'
    # pae = cal.build.load_pae(input_pae,symmetrize=True,colabfold=colabfold) / 10. # nm

    pae = mdp.load_pae(name)

    idx = d_idx//prots_per_row
    jdx = d_idx%prots_per_row
    
    if nprots < prots_per_row:
        axij_pae = ax[2*jdx]
        axij_dmaps = ax[2*jdx+1]
    else:
        axij_pae = ax[idx,2*jdx]
        axij_dmaps = ax[idx,2*jdx+1]
    
    _ = axij_pae.imshow(pae,cmap=plt.cm.Oranges_r,vmin=0,vmax=3.)

    divider = make_axes_locatable(axij_pae)
    cax = divider.append_axes("right", size="5%", pad=0.05)
    # plt.colorbar(im, cax=cax)
    plt.colorbar(_,cax=cax,label='nm')

    axij_pae.set_title(f'{name}\nPAE', fontsize=6, linespacing=1.5)

    nres = len(pae)
    xs = np.arange(0,nres,100)

    cmap = plt.cm.Blues_r

    mapfolder = f'dmaps_std/dmap_{name}.npy'
    dmaps_std = np.load(mapfolder)
    
    _ = axij_dmaps.imshow(dmaps_std,cmap=cmap,vmin=0,vmax=3)
    axij_dmaps.set_title(f'{name}\n'+'$\sigma(r_{ij})$ Sim.', fontsize=6, linespacing=1.5)
    divider = make_axes_locatable(axij_dmaps)
    cax = divider.append_axes("right", size="5%", pad=0.05)
    # plt.colorbar(im, cax=cax)
    plt.colorbar(_,cax=cax,label='nm')

    for a in [axij_pae, axij_dmaps]:
        a.grid(False)
        # a.set(xlabel='Residue',ylabel='Residue')

for idx in range(1,empty+1):
    if nprots < prots_per_row:
        ax[-idx].set_visible(False)
    else:
        ax[-1,-idx].set_visible(False)
        ax[-2,-idx].set(xlabel='Residue')

for idx in range(len(ax)):
    ax[idx,0].set(ylabel='Residue')
    ax[-1,idx].set(xlabel='Residue')

fig.tight_layout(w_pad=0.,h_pad=1)
# fig.savefig(f'figures/pae_vs_dmaps_std_opti_random_tfs.pdf')

## PAE vs sigma(r), single sequence

In [ ]:
mdp = copy.deepcopy(mdp_tf)

mdp.df.set_index('uniprot',inplace=True)

fig, ax = plt.subplots(1,2,figsize=(4,2.2))

name = 'Q99742'

pae = mdp.load_pae(name)

axij_pae = ax[0]

_ = axij_pae.imshow(pae,cmap=plt.cm.Oranges_r,vmin=0,vmax=3.)

divider = make_axes_locatable(axij_pae)
cax = divider.append_axes("right", size="5%", pad=0.05)
# plt.colorbar(im, cax=cax)
plt.colorbar(_,cax=cax,label='nm')

axij_pae.set_title(f'{name}\nPAE', fontsize=6, linespacing=1.5)

nres = len(pae)
xs = np.arange(0,nres,100)

cmap = plt.cm.Blues_r

mapfolder = f'dmaps_std/dmap_{name}.npy'
dmaps_std = np.load(mapfolder)

axij_dmaps = ax[1]

_ = axij_dmaps.imshow(dmaps_std,cmap=cmap,vmin=0,vmax=3)
axij_dmaps.set_title(f'{name}\n'+'$\sigma(r)$ Sim.', fontsize=6, linespacing=1.5)
divider = make_axes_locatable(axij_dmaps)
cax = divider.append_axes("right", size="5%", pad=0.05)
# plt.colorbar(im, cax=cax)
plt.colorbar(_,cax=cax,label='nm')

for a in [axij_pae, axij_dmaps]:
    a.grid(False)
    a.set(xlabel='Residue',ylabel='Residue')

# fig.savefig(f'figures/pae_vs_dmaps_std_opti_{name}.pdf')

# TF IDRs

In [ ]:
@nb.jit(nopython=True)
def find_all_idrs(idr,min_length=30):
    starts = []
    ends = []
    if idr[0] == 1:
        start = 0
        n = 1
    else:
        n = 0
    for idx in range(1, len(idr)):
        if idr[idx] == 1:
            if idr[idx-1] == 0:
                start = idx
                n = 0
            n += 1
        elif idr[idx-1] == 1: # idr[idx] == 0:
            if n >= min_length:# n_longest:
                starts.append(start)
                ends.append(idx-1)
    if idr[-1] == 1:
        if n >= min_length:
            starts.append(start)
            ends.append(idx)

    idr_segments = []
    for start, end in zip(starts, ends):
        idr_segments.append((start+1,end+1)) # 1-based, inclusive
    return idr_segments

## Load TF IDRs

In [ ]:
mdp_idr = MDP_Cyto(
        'tf_idrs',
        csv_file='df_tf_idrs.csv',
        sim_folder='sims_tf_idrs',
        pdb_folder='pdbs_cytosol',
        colabfold=0,
        k_go=15.,
        pae_shift=0.3,
        pae_width=15.,
        bfac_shift=0.8,
        bfac_width=50.,
        cutoff_restraint=0.9,
)

mdp_idr.df.rename({'Unnamed: 0': 'Intracell ID'},axis=1,inplace=True) # ID corresponding to mdp_cyto.df
mdp_idr.df.drop('Unnamed: 0.1', axis=1,inplace=True)

load_backup = False

if load_backup:
    mdp_idr.df = pd.read_csv('mdp_idr_df_backup.csv')
mdp_idr.df

## Calc nu, ete

In [ ]:
start = 10
stop = None
step = 10 # step = 1 in paper

min_key = 0
max_key = None

ete_idrs = []
ete_idrs_in_fl = []

nu_idrs = []
nu_idrs_in_fl = []

n_residues_idrs = []

total_idr_segs = int(np.sum(mdp_idr.df['n_idr_segs'][min_key:max_key]))
print(total_idr_segs)

terminal = np.zeros((total_idr_segs))
central = np.zeros((total_idr_segs))

ct = 0

nu_large_diffs = []

for idx, val in mdp_idr.df[min_key:max_key].iterrows():
    print(idx)
    name = val['uniprot']
    
    u_fl = mda.Universe(f'{mdp_tf.sim_folder}/{name}/top.pdb', f'{mdp_tf.sim_folder}/{name}/{name}.dcd')

    n_residues_fl = len(val['seq'])

    if math.isnan(float(val['n_idr_segs'])):
    #     print(name)
        continue
    for jdx in range(int(val['n_idr_segs'])):
        idr_start, idr_end  = int(val[f'start_{jdx}']), int(val[f'end_{jdx}']) # 1 based inclusive
        path = f'{mdp_idr.sim_folder}/{name}_{idr_start}_{idr_end}'

        n_residues_idr = idr_end - idr_start + 1
        n_residues_idrs.append(n_residues_idr)

        if (idr_start == 1) or (idr_end == n_residues_fl):
            terminal[ct] = 1
        else:
            central[ct] = 1

        u_idr = mda.Universe(f'{path}/top.pdb', f'{path}/{name}_{idr_start}_{idr_end}.dcd')
        ag_idr = u_idr.atoms
        _, ete_idr, _ = cal.analysis.calc_ete(u_idr, ag_idr, start=start, stop=stop, step=step)
        ete_idr /= np.sqrt(n_residues_idr)

        ag_idr_in_fl = u_fl.atoms[idr_start-1:idr_end]
        _, ete_idr_in_fl, _ = cal.analysis.calc_ete(u_fl, ag_idr_in_fl, start=start, stop=stop, step=step)
        ete_idr_in_fl /= np.sqrt(n_residues_idr)

        ij, dij, _, nu_idr, nu_err = cal.analysis.fit_scaling_exp(u_idr,ag_idr,start=start,stop=stop,step=step)
        ij, dij, _, nu_idr_in_fl, nu_err = cal.analysis.fit_scaling_exp(u_fl,ag_idr_in_fl,start=start,stop=stop,step=step)

        if abs(nu_idr-nu_idr_in_fl) > 0.2:
            print(f'{name}_{idr_start}_{idr_end}')
            nu_large_diffs.append(idx)

        mdp_idr.df.loc[idx,f'ete_idr_{jdx}'] = ete_idr
        mdp_idr.df.loc[idx,f'ete_idr_in_fl_{jdx}'] = ete_idr_in_fl
        mdp_idr.df.loc[idx,f'nu_idr_{jdx}'] = nu_idr
        mdp_idr.df.loc[idx,f'nu_idr_in_fl_{jdx}'] = nu_idr_in_fl

        ete_idrs.append(ete_idr)
        ete_idrs_in_fl.append(ete_idr_in_fl)

        nu_idrs.append(nu_idr)
        nu_idrs_in_fl.append(nu_idr_in_fl)

        ct += 1

mdp_idr.df.to_csv('mdp_idr_df_backup.csv')

nu_large_diffs = np.array(nu_large_diffs)
ete_idrs = np.array(ete_idrs)
ete_idrs_in_fl = np.array(ete_idrs_in_fl)
nu_idrs = np.array(nu_idrs)
nu_idrs_in_fl = np.array(nu_idrs_in_fl)
n_residues_idrs = np.array(n_residues_idrs)

In [ ]:
titles = [
    'All IDRs',
    'Terminal IDRs',
    'Central IDRs',
]

labels = [
    'Isolated',
    'In MDP context',
]

colors = [
    'black',
    'C0',
    'C1'
]

both = np.arange(len(ete_idrs))
ter = np.where(terminal)[0]
cent = np.where(central)[0]

In [ ]:
# Scatter plots

fig, ax = plt.subplots(2,3,figsize=(5,3.5),sharey='row')#,sharex=True)

for idx, selection in enumerate([both,ter,cent]):
    axij = ax[0,idx]
    xs = np.arange(0,2,0.1)
    axij.plot(ete_idrs[selection], ete_idrs_in_fl[selection],'.',markersize=1.3)
    axij.plot(xs,xs,color='black',lw=0.5)
    axij.set_title(titles[idx],fontsize=6)
    axij.set(xlabel=r'$\tilde{R}_\mathrm{ee}$'+' isolated [nm]')
    axij.set(xlim=(0.1,1.2), ylim=(0.1,1.2))
    axij.set(yticks=(0.25,0.5,0.75,1.0))

    p = pearsonr(ete_idrs[selection], ete_idrs_in_fl[selection])
    sp = spearmanr(ete_idrs[selection], ete_idrs_in_fl[selection])
    axij.text(0.17,1.1,f'$r={p.statistic:.2f}$, '+r'$\rho=$'+f'{sp.statistic:.2f}')
    
    axij = ax[1,idx]
    xs = np.arange(-1,1,0.1)
    axij.plot(nu_idrs[selection], nu_idrs_in_fl[selection],'.',markersize=1.3)
    axij.plot(xs,xs,color='black',lw=0.5)
    axij.set_title(titles[idx],fontsize=6)
    axij.set(xlabel=r'$\nu$'+' isolated')
    axij.set(xlim=(-0.1,0.75), ylim=(-0.1,0.75))

    p = pearsonr(nu_idrs[selection], nu_idrs_in_fl[selection])
    sp = spearmanr(nu_idrs[selection], nu_idrs_in_fl[selection])
    axij.text(-0.05,0.66,f'$r={p.statistic:.2f}$, '+r'$\rho=$'+f'{sp.statistic:.2f}')

ax[0,0].set(ylabel=r'$\tilde{R}_\mathrm{ee}$'+' in MDP [nm]')
ax[1,0].set(ylabel=r'$\nu$'+' in MDP')

fig.tight_layout(w_pad=1)
# fig.savefig('figures/expansion_idr_in_context_scatter.pdf')

In [ ]:
def bin_data(xs,ys,nbins,drange=None):
    """ bin data ys in xs bins, based on numpy.histogram_bin_edges """
    if drange == None:
        xmin, xmax = np.min(xs), np.max(xs)
    else:
        xmin, xmax = drange[0], drange[1]
    bins = np.linspace(xmin,xmax,nbins+1)
    y_binned = [[] for _ in range(nbins)]
    for x, y in zip(xs,ys):
        if x <= bins[0]:
            y_binned[0].append(y)
        elif x >= bins[-1]:
            y_binned[nbins-1].append(y)
        else:
            for idx in range(nbins):
                if x >= bins[idx] and x < bins[idx+1]:
                    y_binned[idx].append(y)
    return bins, y_binned

In [ ]:
drange = (30,500)

nwindows = 8

e, idrs_win_ete = bin_data(n_residues_idrs, ete_idrs, nwindows, drange=drange)
e, idrs_in_fl_win_ete = bin_data(n_residues_idrs, ete_idrs_in_fl, nwindows, drange=drange)

e, idrs_win_nu = bin_data(n_residues_idrs, nu_idrs, nwindows, drange=drange)
e, idrs_in_fl_win_nu = bin_data(n_residues_idrs, nu_idrs_in_fl, nwindows, drange=drange)

In [ ]:
fig, ax = plt.subplots(2,nwindows,figsize=(nwindows*1.3,3.),sharey=False)
for idx in range(nwindows):
    # ete

    axij = ax[0,idx]

    xs = np.arange(0,2,0.1)
    axij.plot(idrs_win_ete[idx], idrs_in_fl_win_ete[idx],'.',markersize=1.3)
    axij.plot(xs,xs,color='black',lw=0.5)

    axij.set(xlabel=r'$\tilde{R}_\mathrm{ee}$'+' isolated')
    # axij.set(xlabel=r'$<R_\mathrm{ee}>/\sqrt{N}$')
    axij.set(xlim=(0.1,1.2), ylim=(0.1,1.2))

    # nu
    axij = ax[1,idx]

    xs = np.arange(-1,1,0.1)
    axij.plot(idrs_win_nu[idx], idrs_in_fl_win_nu[idx],'.',markersize=1.3)
    axij.plot(xs,xs,color='black',lw=0.5)
    
    axij.set(xlabel=r'$\nu$'+' isolated')
    axij.set(xlim=(-0.1,0.75), ylim=(-0.1,0.75))
    # axij.set_yscale('log')

    for jdx in range(2):
        if idx == nwindows-1:
            ax[jdx,idx].set_title(r'$N \geq$'+f'{int(e[idx])}')
        else:
            ax[jdx,idx].set_title(f'{int(e[idx])}'+r'$\leq N \less$'+f'{int(e[idx+1])}')

ax[0,0].set(ylabel=r'$\tilde{R}_\mathrm{ee}$'+' in MDP')
ax[1,0].set(ylabel=r'$\nu$'+' in MDP')

fig.tight_layout()#w_pad=0)
# fig.savefig('figures/expansion_idr_vs_nres.pdf')

## Folded domains around IDRs

In [ ]:
min_key = 0
max_key = None

zone = 20

for idx, val in tqdm(mdp_idr.df[min_key:max_key].iterrows()):
    # print('-----')
    name = val['uniprot']
    # print(val['seq'])
    N = len(val['seq'])
    mapfolder = f'dmaps_std/dmap_{name}.npy'
    dmaps_std = np.load(mapfolder)
    for jdx in range(9):
        start, end = val[f'start_{jdx}']-1, val[f'end_{jdx}'] # 0-based exclusive
        if np.isnan(start) or np.isnan(end):
            continue
        start, end = int(start), int(end)
        # print(start,end)
        # print(start < zone, N-end < zone)
        if (start < zone) or (N-end < zone):
            continue
        dmaps_cross = dmaps_std[start-zone:start,end:end+zone]
        # print(np.mean(dmaps_cross))
        mdp_idr.df.loc[idx,f'dmaps_cross_{jdx}'] = np.mean(dmaps_cross)

In [ ]:
delta_nus = []
delta_etes = []
dmaps = []

for idx, val in tqdm(mdp_idr.df[min_key:max_key].iterrows()):
    # print('-----')
    name = val['uniprot']
    N = len(val['seq'])
    for jdx in range(9):
        if not np.isnan(val[f'dmaps_cross_{jdx}']):
            delta_nu = val[f'nu_idr_in_fl_{jdx}'] - val[f'nu_idr_{jdx}']
            delta_nus.append(delta_nu)
            delta_ete = val[f'ete_idr_in_fl_{jdx}'] - val[f'ete_idr_{jdx}']
            delta_etes.append(delta_ete)
            dmaps.append(val[f'dmaps_cross_{jdx}'])

            if delta_nu < -0.2:
                print(f'{name}, {N}, {jdx}, {delta_ete:.2f}, {delta_nu:.2f}, {val[f"start_{jdx}"]}, {val[f"end_{jdx}"]}')

In [ ]:
# Examples
# Q99742 stretch 200-242 loop of the same domain https://www.cathdb.info/version/v4_4_0/cathnode/3.30.450.20

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(4,1.7))
ax[0].plot(delta_etes,dmaps,'.',markersize=1.3)
ax[0].set(xlabel=r'$\Delta\tilde{R}_\mathrm{ee}$ [nm]')
ax[1].plot(delta_nus,dmaps,'.',markersize=1.3)
ax[1].set(xlabel=r'$\Delta\nu$')
for idx in range(2):
    ax[idx].set(ylabel=r'$\sigma(r)$ (flanking domains) [nm]')
# fig.savefig('figures/expansion_vs_surr_folded_sigma.pdf')

## Scatter plot with IDR features

In [ ]:
residues = pd.read_csv('residues_CALVADOS3.csv').set_index('one')
aminoacids = "ACDEFGHIKLMNPQRSTVWY"
nu_file = 'svr_model_nu.joblib'

In [ ]:
delta_nus = []
delta_etes = []

lambda_map = cal.sequence.make_lambda_map(residues)
ah_intgrl_map = cal.sequence.make_ah_intgrl_map(residues)

features_idrs = []

exclude_strong_flanks = True

for key, val in tqdm(mdp_idr.df.iterrows(),total=len(mdp_idr.df)):
    seq = val['seq']
    # print(seq)
    if np.isnan(val['n_idr_segs']):
        continue
    n_idrs = int(val['n_idr_segs'])
    for idx in range(n_idrs):
        if exclude_strong_flanks and (f'dmaps_cross_{idx}' in mdp_idr.df.columns):
            if not np.isnan(val[f'dmaps_cross_{idx}']):
                if val[f'dmaps_cross_{idx}'] < 1: # exclude idrs with strongly interacting flanking domains
                    print(f'Excluding {key} idr {idx}')
                    continue
        start = int(val[f'start_{idx}'])
        end = int(val[f'end_{idx}'])

        if (start == 1) and (end == len(seq)):
            charge_termini = 'both'
        elif start == 1:
            charge_termini = 'N'
        elif end == len(seq):
            charge_termini = 'C'
        else:
            charge_termini = 'none'

        seq_idr = seq[start-1:end]
        # print(len(seq_idr))

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            features = cal.sequence.SeqFeatures(seq_idr,residues=residues,charge_termini=charge_termini,
                                            nu_file=nu_file,ah_intgrl_map=ah_intgrl_map,
                                            lambda_map=lambda_map)
            features_idrs.append(features)
        delta_nu = val[f'nu_idr_in_fl_{idx}'] - val[f'nu_idr_{idx}']
        delta_nus.append(delta_nu)
        delta_ete = val[f'ete_idr_in_fl_{idx}'] - val[f'ete_idr_{idx}']
        delta_etes.append(delta_ete)
delta_nus = np.array(delta_nus)
delta_etes = np.array(delta_etes)
    # print(seq, n_idrs)
    # for idx in range(int(val['n_idr_segs'])):
        # print(idx)

In [ ]:
def bin_data(xs,ys,nbins,drange=None):
    """ bin data ys in xs bins, based on numpy.histogram_bin_edges """
    if drange == None:
        xmin, xmax = np.min(xs), np.max(xs)
    else:
        xmin, xmax = drange[0], drange[1]
    bins = np.linspace(xmin,xmax,nbins+1)
    y_binned = [[] for _ in range(nbins)]
    for x, y in zip(xs,ys):
        if x <= bins[0]:
            y_binned[0].append(y)
        elif x >= bins[-1]:
            y_binned[nbins-1].append(y)
        else:
            for idx in range(nbins):
                if x >= bins[idx] and x < bins[idx+1]:
                    y_binned[idx].append(y)
    return bins, y_binned

features_clean = {
    'mean_lambda' : r'$\bar{\lambda}$',
    'faro' : r'$f_\mathrm{aro}$',
    'shd' : 'SHD',
    'ncpr' : 'NCPR',
    'fcr' : 'FCR',
    'scd' : 'SCD',
    'ah_ij' : r'AH$_\mathrm{pairs}$',
    'q_ij' : r'q$_\mathrm{pairs}$',
    'nu_svr' : r'$\nu_\mathrm{SVR}$',
    'N' : 'N',
    'kappa' : r'$\kappa$',
    'mw' : r'M$_w$',
    'dG' : '$\Delta G$ [$k_\mathrm{B}T$]',
    'width_interf' : 'Interface width [nm]',
    'rg_rel' : r'$\overline{R}_{g,\text{Interface}}$' + ' / ' +r'$\overline{R}_{g,\text{Dense}}$',
    'sz' : r'$S_z$ (Interface)',
    'seqid' : 'Seq. ID',
    'seqJS' : 'JS Div.',
}

targets_clean = {
    'dG' : '$\Delta G$ [$k_\mathrm{B}T$]',
    'logcdil_mgml' : 'ln($c_\mathrm{dil}$ [g/L])',
    'log10cdil_mgml' : r'log$_{10}$($c_\mathrm{dil}$ [g/L])',
    'dG_pred_error' : r'$\Delta G$ Error [$k_\mathrm{B}T$]',
}

mltypes_clean = {
    'svr' : 'SVR',
    'mlp' : 'Dense NN',
}

units_clean = {
    'dG' : r'$k_\mathrm{B}T$',
    'logcdil_mgml' : ''
}

limits = {
    'scd' : [-5,6],
    # 'scd' : [-8,6],
    'ncpr' : [-0.25, 0.25],
    'kappa' : [0.0, 0.8],
    'faro' : [0., 0.15],
    'nu' : [0.2, 0.7],
    'nu_svr' : [0.45, 0.62],
    'mw' : [0., 60000],
    'fcr' : [0., 0.6],
    'mean_lambda' : [0.3, 0.6],
    'shd' : [1.5, 6.],
    'N' : [0, 800],
    'ah_ij' : [-0.8,-0.3],
}

In [ ]:
# feat = 'ncpr'
feats = ['mean_lambda', 'faro', 'shd', 'ncpr', 'fcr', 'scd', 'ah_ij','nu_svr']#'nu_svr'] #

# fig, ax = plt.subplots(len(feats),1,figsize=(3,9))

col1x = 3.425 # column
col2x = 7. # 2x column

nbins = 100
drange = None

fig, ax = plt.subplots(2,4,figsize=(col2x,2.5),sharey=True)

expansion_prop = 'delta_etes' # 'delta_nus' # 

if expansion_prop == 'delta_nus':
    ys = delta_nus
    ylabel = r'$\Delta\nu$'
    ylim=(-0.07,0.07)
    
elif expansion_prop == 'delta_etes':
    ys = delta_etes
    # ylabel = r'$\Delta(<R_\mathrm{ee}>/\sqrt{N})$'
    ylabel = r'$\Delta\tilde{R}_\mathrm{ee}$ [nm]'
    ylim = (-0.1,None)

for idx, feat in enumerate(feats):
    xs = []
    
    # axij = ax[idx]
    axij = ax[idx//4,idx%4]
    
    for features in features_idrs:
        xs.append(getattr(features,feat))
    xs = np.array(xs)

    if feat in limits:
        drange = limits[feat]
    else:
        drange = None
    
    edges, y_binned = bin_data(xs,ys,nbins,drange=drange)
    edges = (edges[:-1] + edges[1:]) / 2.

    # print(edges)
    y_mean = np.array([np.mean(y) for y in y_binned])
    y_std = np.array([np.std(y) for y in y_binned])
    yLs = np.array([len(yb) for yb in y_binned])

    for e, ym, yst, yL in zip(edges,y_mean,y_std,yLs):
        # color = fcolor_r(min(0.5,yL/(1.05*max(yLs))))
        color = colors[0]
        markers, caps, bars = axij.errorbar(e,ym,yerr=yst,capsize=1,marker='.',color='C0',zorder=10)
    # loop through bars and  caps and set the alpha value
        [bar.set_alpha(0.3) for bar in bars]
        [cap.set_alpha(0.3) for cap in caps]
    
    axij.set(xlabel=features_clean[feat])
    axij.set(ylabel=ylabel)
    axij.set(ylim=ylim)
# fig.savefig(f'figures/{expansion_prop}_vs_idr_features.pdf')